# Tracking Subgroup Metadata Preparation

This notebook builds the Tracking-cohort equivalent of `subgroups.csv` —
a per-patient-per-visit metadata table (subgroup label, gender, age at
visit) that other Tracking-cohort notebooks can join against, mirroring
the PPMI/OPDC `subgroups.csv` schema.

**What this notebook does:**
1. Loads the raw Tracking subgroup assignments and relabels them to
   match the PPMI/OPDC column convention.
2. Excludes monogenic (known-genetic-cause) PD patients, keeping only
   the idiopathic-PD cohort used elsewhere.
3. Loads and merges in gender from the Tracking baseline table.
4. Loads and merges in age-at-visit from the Tracking longitudinal table.
5. Writes the combined table to `data/02_processed/TRACKING/subgroups.csv`.

In [1]:
import pandas as pd
import pickle

### Load Subgroup Assignments

In [2]:
subgroups = pd.read_csv('./../../data/01_raw/TRACKING/subgroups_raw.csv')

In [3]:
# Sanity check: raw subgroup counts, as recorded in the source column.
subgroups['C4XD subgroup'].value_counts()

C4XD subgroup
B    896
A    539
C    377
Name: count, dtype: int64

In [4]:
# Sanity check: same counts, as a proportion of the full table.
subgroups['C4XD subgroup'].value_counts() / subgroups.shape[0]

C4XD subgroup
B    0.494481
A    0.297461
C    0.208057
Name: count, dtype: float64

In [5]:
# Rename to match the PPMI/OPDC subgroup-table column convention.
subgroups = subgroups.rename(columns={'participant_id': 'PATNO', 'C4XD subgroup': 'Group'})

In [6]:
# Recode subgroup labels from letters to numbers (see note above about
# this being the opposite direction from the PPMI/OPDC notebooks).
subgroups['Group'] = subgroups['Group'].replace('A', 0)
subgroups['Group'] = subgroups['Group'].replace('B', 1)
subgroups['Group'] = subgroups['Group'].replace('C', 2)

/var/folders/5v/vtmbr3255f59jlqhnxc_l2440000gp/T/ipykernel_94971/1135395450.py:5: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  subgroups['Group'] = subgroups['Group'].replace('C', 2)


### Exclude Monogenic PD Patients

Patients with a known monogenic (single-gene) cause of PD are excluded here, keeping only the idiopathic-PD cohort used by the rest of the pipeline.

In [7]:
monogenic = pd.read_csv('./../../data/01_raw/TRACKING/monogenic.csv')

In [8]:
monogenic

,participant_id,rsID,GT
0,ABED,rs2230288,1
1,AGIO,rs2230288,1
2,AMYL,rs2230288,1
3,APER,rs2230288,1
4,AULD,rs2230288,1
...,...,...,...
79,TOYS,rs2230288,1
80,TRON,rs2230288,1
81,TWAY,rs2230288,1
82,WENS,rs2230288,1


In [9]:
subgroups

,PATNO,Group
0,AAHS,1
1,AALS,1
2,ABAS,1
3,ABBA,2
4,ABBE,1
...,...,...
1807,WYES,2
1808,WYLE,0
1809,WYNS,0
1810,YEAR,1


In [10]:
subgroups = subgroups[~subgroups['PATNO'].isin(monogenic['participant_id'])].reset_index(drop=True)

In [11]:
subgroups

,PATNO,Group
0,AAHS,1
1,AALS,1
2,ABAS,1
3,ABBA,2
4,ABBE,1
...,...,...
1723,WUDS,1
1724,WYES,2
1725,WYLE,0
1726,WYNS,0


### Add Gender

In [12]:
gender = pd.read_csv('./../../data/01_raw/TRACKING/P3_Tracking_Baseline.csv')[['ID', 'gender']]
gender['PATNO'] = gender['ID']
gender.drop(columns='ID', inplace=True)

# Recode to PPMI's numeric GENDER convention (1 = male, 0 = female).
# Note: Tracking's raw labels are capitalized ('Male'/'Female'), unlike
# OPDC's lowercase 'male'/'female' — case matters for these replacements.
gender['gender'] = gender['gender'].replace('Male', 1)
gender['gender'] = gender['gender'].replace('Female', 0)
gender.rename(columns={'gender': 'GENDER'}, inplace=True)

/var/folders/5v/vtmbr3255f59jlqhnxc_l2440000gp/T/ipykernel_94971/2810802327.py:9: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  gender['gender'] = gender['gender'].replace('Female', 0)


In [13]:
gender

,GENDER,PATNO
0,1,AAHS
1,0,AALS
2,0,ABAS
3,1,ABBA
4,1,ABBE
...,...,...
1953,1,WYND
1954,1,WYNN
1955,1,WYNS
1956,1,YEAR


In [14]:
subgroups = pd.merge(subgroups, gender, on='PATNO')
subgroups

,PATNO,Group,GENDER
0,AAHS,1,1
1,AALS,1,0
2,ABAS,1,0
3,ABBA,2,1
4,ABBE,1,1
...,...,...,...
1687,WUDS,1,0
1688,WYES,2,1
1689,WYLE,0,1
1690,WYNS,0,1


### Add Age at Visit

In [15]:
age = pd.read_csv('./../../data/01_raw/TRACKING/P3_Tracking_Longitudinal_2025.csv')[['ID', 'Visit', 'age_at_visit']]
age.rename(columns={'ID': 'PATNO', 'Visit': 'EVENT_ID', 'age_at_visit': 'AGE_AT_VISIT'}, inplace=True)

# Tracking visit numbers are cast to string here, matching the EVENT_ID
# typing convention used in 05_Tracking_Data_Prep.ipynb's Part III table.
age['EVENT_ID'] = age['EVENT_ID'].astype(str)

In [16]:
age

,PATNO,EVENT_ID,AGE_AT_VISIT
0,AAHS,1,55.277206
1,AAHS,4,56.807667
2,AAHS,7,58.381931
3,AAHS,9,59.682411
4,AAHS,10,61.719372
...,...,...,...
6911,YEAR,4,57.563313
6912,YEAR,7,59.104721
6913,YEAS,1,66.997948
6914,YEAS,4,68.607803


In [17]:
# Merging on PATNO alone (not PATNO + EVENT_ID) broadcasts every visit's
# age against each patient's (Group, GENDER) row, producing one output
# row per patient-visit.
subgroups = pd.merge(subgroups, age, on='PATNO')
subgroups

,PATNO,Group,GENDER,EVENT_ID,AGE_AT_VISIT
0,AAHS,1,1,1,55.277206
1,AAHS,1,1,4,56.807667
2,AAHS,1,1,7,58.381931
3,AAHS,1,1,9,59.682411
4,AAHS,1,1,10,61.719372
...,...,...,...,...,...
6025,WYNS,0,1,4,58.475018
6026,WYNS,0,1,7,59.969883
6027,YEAR,1,1,1,56.038330
6028,YEAR,1,1,4,57.563313


In [18]:
subgroups = subgroups.sort_values(["PATNO", "EVENT_ID"]).reset_index(drop=True)

# Years elapsed since the previous visit for this subject (NaN on each
# subject's first visit, since there's no prior visit to diff against).
subgroups["visit_interval_years"] = (subgroups.groupby("PATNO")["AGE_AT_VISIT"].diff())

# Cumulative years since the subject's first visit: treat the first
# visit's undefined interval as 0, then cumulatively sum the intervals.
subgroups["time_since_first_visit"] = (
    subgroups.groupby("PATNO")["visit_interval_years"].transform(lambda x: x.fillna(0).cumsum())
)

In [20]:
subgroups = subgroups.drop(columns = ['EVENT_ID' , 'visit_interval_years']).rename(columns={'time_since_first_visit' : 'EVENT_ID'})

### Export

In [22]:
subgroups.to_csv('./../../data/02_processed/TRACKING/subgroups.csv')